# 8. Tool Access and Authorization: Identity Is Not Enough

By this point in the repo, we have seen how a user can authenticate with a trusted identity provider, receive a signed token, and verify who they are. This notebook asks a different question:

- Is this user allowed to invoke this tool?
- Can they use it on this exact resource?
- What happens when the identity is valid but the authorization check fails?

The short version is simple: **authentication proves identity; authorization decides permission**.


## 8.1 The core distinction

A user can be authenticated and still not be allowed to do something.

That is not a bug in the identity system. It is exactly how real access control is supposed to work.

Examples:

- `alice` can log in successfully, but cannot delete a note
- `bob` can log in and edit notes, but cannot deploy production
- `charlie` can authenticate, but has no tool access at all
- `admin` can do more than normal users

A real system usually checks **both**:

1. Are you who you claim to be?
2. Are you allowed to invoke this tool on this resource?


In [1]:
# A tiny simulation of authenticated users and their roles.
# The important thing here is not the exact syntax. It is the pattern.

users = {
    "alice": {"roles": ["reader"], "projects": ["team-alpha"]},
    "bob": {"roles": ["editor"], "projects": ["team-alpha", "team-beta"]},
    "charlie": {"roles": [], "projects": []},
    "admin": {"roles": ["admin"], "projects": ["*"]},
}

# Tool permissions are separate from identity.
tool_permissions = {
    "read_note": {"reader", "editor", "admin"},
    "write_note": {"editor", "admin"},
    "delete_note": {"admin"},
    "deploy_release": {"admin"},
}

for username in users:
    print(f"{username}: roles={users[username]['roles']}")
print()
print("Tool permissions")
for tool, roles in tool_permissions.items():
    print(f"- {tool}: {sorted(roles)}")


alice: roles=['reader']
bob: roles=['editor']
charlie: roles=[]
admin: roles=['admin']

Tool permissions
- read_note: ['admin', 'editor', 'reader']
- write_note: ['admin', 'editor']
- delete_note: ['admin']
- deploy_release: ['admin']


## 8.2 A policy function

We can model authorization as a function that checks whether a principal is allowed to call a tool.

This is intentionally simple, but it follows the same pattern used in real policy engines:

- identity is known
- tool and resource are known
- policy decides whether access is permitted
- if denied, the system returns a clear authorization failure


In [2]:
def can_call_tool(username: str, tool: str) -> bool:
    """Return True if the user is allowed to call the tool."""
    if username not in users:
        return False

    user_roles = set(users[username]["roles"])
    allowed_roles = tool_permissions.get(tool, set())
    return bool(user_roles & allowed_roles)

for tool in ["read_note", "write_note", "delete_note", "deploy_release"]:
    print(f"{tool}:")
    for username in ["alice", "bob", "charlie", "admin"]:
        print(f"  - {username}: {can_call_tool(username, tool)}")


read_note:
  - alice: True
  - bob: True
  - charlie: False
  - admin: True
write_note:
  - alice: False
  - bob: True
  - charlie: False
  - admin: True
delete_note:
  - alice: False
  - bob: False
  - charlie: False
  - admin: True
deploy_release:
  - alice: False
  - bob: False
  - charlie: False
  - admin: True


Notice the shape of the result:

- `alice` is authenticated, but she is not in the role set for `delete_note`
- `bob` can read and write, but not deploy
- `charlie` has no roles, so the system denies access
- `admin` is allowed to do the administrative actions

This is the key conceptual idea: a valid identity does not grant blanket authority.


In [3]:
# A more realistic version adds a resource: the specific project or document.
# The same user may be allowed to read one project but not another.

permissions = {
    "alice": {
        "team-alpha": {"read_note": True, "write_note": False, "delete_note": False},
    },
    "bob": {
        "team-alpha": {"read_note": True, "write_note": True, "delete_note": False},
        "team-beta": {"read_note": True, "write_note": True, "delete_note": False},
    },
    "charlie": {
        "team-alpha": {"read_note": False, "write_note": False, "delete_note": False},
    },
    "admin": {
        "team-alpha": {"read_note": True, "write_note": True, "delete_note": True},
        "team-beta": {"read_note": True, "write_note": True, "delete_note": True},
    },
}

def can_call_tool_on_resource(username: str, tool: str, resource: str) -> bool:
    user_access = permissions.get(username, {})
    return bool(user_access.get(resource, {}).get(tool, False))

for username in ["alice", "bob", "charlie", "admin"]:
    print(f"\nUser: {username}")
    for tool in ["read_note", "write_note", "delete_note"]:
        print(f"  {tool} on team-alpha -> {can_call_tool_on_resource(username, tool, 'team-alpha')}")



User: alice
  read_note on team-alpha -> True
  write_note on team-alpha -> False
  delete_note on team-alpha -> False

User: bob
  read_note on team-alpha -> True
  write_note on team-alpha -> True
  delete_note on team-alpha -> False

User: charlie
  read_note on team-alpha -> False
  write_note on team-alpha -> False
  delete_note on team-alpha -> False

User: admin
  read_note on team-alpha -> True
  write_note on team-alpha -> True
  delete_note on team-alpha -> True


## 8.3 A tool gateway

This is the boundary where a real system enforces access. The token may prove who the user is, but the tool gateway checks whether that user can actually do the action.

The shape is often the same in production:

1. Verify the identity token
2. Extract the subject
3. Read the requested tool and resource
4. Check policy
5. Execute or deny
6. Log the decision


In [4]:
def authenticate_user(username: str) -> dict:
    """Pretend we have verified a signed identity token and extracted the subject."""
    if username not in users:
        raise ValueError(f"Unknown user: {username}")
    return {"sub": username, "roles": users[username]["roles"]}

def tool_gateway(username: str, tool: str, resource: str) -> str:
    """A toy gateway that enforces the final authorization decision."""
    principal = authenticate_user(username)
    if not can_call_tool_on_resource(principal["sub"], tool, resource):
        return f"DENIED: {principal['sub']} is not allowed to call {tool} on {resource}"
    return f"ALLOWED: {principal['sub']} may call {tool} on {resource}"

for request in [
    ("alice", "read_note", "team-alpha"),
    ("alice", "delete_note", "team-alpha"),
    ("bob", "write_note", "team-beta"),
    ("bob", "deploy_release", "team-beta"),
    ("admin", "deploy_release", "team-alpha"),
]:
    print(tool_gateway(*request))


ALLOWED: alice may call read_note on team-alpha
DENIED: alice is not allowed to call delete_note on team-alpha
ALLOWED: bob may call write_note on team-beta
DENIED: bob is not allowed to call deploy_release on team-beta
DENIED: admin is not allowed to call deploy_release on team-alpha


## 8.4 Why this is the production shape

This demonstration is intentionally tiny, but it reflects the real production pattern behind tools, APIs, and agent platforms:

- The system first proves who the caller is
- The tool boundary then asks whether that caller is allowed to invoke the requested action
- A denial is a clear, intentional access decision, not a vague failure
- Access control is often resource-aware and action-aware

In other words, identity tells us **who** the user is. Policy tells us **what they are allowed to do**.

That is the key conceptual bridge from login to tool access.
